# Kafka 风险告警消费

消费交易事件，识别大额交易与支付失败。先用本地数据验证规则。

## 运行条件

需要安装 `confluent-kafka`，并准备可访问的 Kafka 和目标 Topic。本地示例应返回 `high_amount` 和 `payment_failed`。将 `start_consumer` 改为 `True` 后持续消费，使用中断内核停止；`finally` 会关闭消费者。


In [ ]:
from __future__ import annotations
import argparse
import json
from typing import Any


## 实现


In [ ]:
def risk_reasons(event: dict[str, Any], amount_threshold: float) -> list[str]:
    reasons = []
    amount = float(event.get("amount", 0))
    if amount >= amount_threshold:
        reasons.append("high_amount")
    if event.get("event_type") == "payment_result" and event.get("status") == "PAY_FAILED":
        reasons.append("payment_failed")
    return reasons


In [ ]:
def consume_events(bootstrap_servers: str, topic: str, group_id: str) -> None:
    from confluent_kafka import Consumer, KafkaError

    consumer = Consumer(
        {
            "bootstrap.servers": bootstrap_servers,
            "group.id": group_id,
            "enable.auto.commit": False,
            "auto.offset.reset": "earliest",
            "client.id": "processing-demo-consumer",
        }
    )

    consumer.subscribe([topic])
    try:
        while True:
            message = consumer.poll(timeout=1.0)
            if message is None:
                continue
            if message.error():
                if message.error().code() == KafkaError._PARTITION_EOF:
                    continue
                raise RuntimeError(message.error())

            event = json.loads(message.value().decode("utf-8"))
            reasons = risk_reasons(event, amount_threshold=3000.0)
            if reasons:
                alert = {
                    "order_id": event.get("order_id"),
                    "shop_id": event.get("shop_id"),
                    "amount": event.get("amount"),
                    "reasons": reasons,
                }
                print(json.dumps(alert, ensure_ascii=False))
            consumer.commit(message=message, asynchronous=False)
    finally:
        consumer.close()


## 本地检查

此单元不连接外部服务。


In [ ]:
event = {"amount": 5200, "event_type": "payment_result", "status": "PAY_FAILED"}
risk_reasons(event, amount_threshold=3000.0)


## 连接服务

先修改地址与参数，再开启执行开关。


In [ ]:
bootstrap_servers = "localhost:9092"
topic = "trade_events"
group_id = "risk-online-group"
start_consumer = False

if start_consumer:
    consume_events(bootstrap_servers, topic, group_id)


## 命令行入口

导出为脚本后使用；在 Notebook 内运行仅定义入口，不解析内核参数。


In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--bootstrap", required=True)
    parser.add_argument("--topic", default="trade_events")
    parser.add_argument("--group-id", default="risk-online-group")
    args = parser.parse_args()
    consume_events(args.bootstrap, args.topic, args.group_id)


if __name__ == "__main__" and "get_ipython" not in globals():
    main()
